In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.util, json, subprocess, sys, time
import numpy as np
import pandas as pd

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
if importlib.util.find_spec("numpyro") is None:
    print("Installing NumPyro without changing Kaggle's JAX/CUDA packages...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "numpyro==0.22.0"], check=True)
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
from numpyro.infer.initialization import init_to_value
from numpyro.diagnostics import summary as numpyro_summary
import matplotlib.pyplot as plt

print("JAX backend:", jax.default_backend(), "| devices:", jax.devices(), "| NumPyro:", numpyro.__version__)
if jax.default_backend() != "gpu":
    raise RuntimeError("GPU preflight failed: select GPU T4 x2, restart the session, then run again.")

binary_sources=[]
for mp in INPUT.rglob("simulation_metadata.json"):
    if mp.parent.name != "binary":
        continue
    try:
        meta=json.loads(mp.read_text(encoding="utf-8"))
    except Exception:
        continue
    if meta.get("family")=="binary" and int(meta.get("parameter_draws_unique",0))>=100_000:
        binary_sources.append((meta.get("created_utc",""),mp.parent))
if not binary_sources:
    raise FileNotFoundError("Attach Day 7 Cosmic swan Bayesian binary simulation output.")
_, BINARY_DIR=sorted(binary_sources,key=lambda x:x[0])[-1]
observations=np.load(BINARY_DIR/"observations.npy",mmap_mode="r")
parameters=pd.read_csv(BINARY_DIR/"parameters.csv")
time_days=np.asarray(np.load(BINARY_DIR/"time_days.npy"),dtype=float)
if observations.shape != (len(parameters),len(time_days)):
    raise ValueError("Day 7 observations, parameters, and time axis do not match.")

def newest(pattern,label):
    paths=list(INPUT.rglob(pattern))
    if not paths:
        raise FileNotFoundError(f"Attach saved output containing {label}: {pattern}")
    return max(paths,key=lambda p:p.stat().st_mtime)

NUTS_PATH=newest("day9_9_nuts_case_diagnostics.csv","Day 9.9 NUTS diagnostics")
MAP_PARAMS_PATH=newest("day9_9_map_parameter_recovery.csv","Day 9.9 MAP parameter estimates")
PROFILE_PATH=newest("day9_10_period_profile_summary.csv","Day 9.10 profile summary")
nuts=pd.read_csv(NUTS_PATH)
map_params=pd.read_csv(MAP_PARAMS_PATH)
profiles=pd.read_csv(PROFILE_PATH)
CASE_ROWS=sorted(nuts.loc[nuts.status.astype(str).ne("screen_pass"),"simulation_row"].astype(int))
if not CASE_ROWS:
    raise ValueError("Day 9.9 contains no NUTS review rows.")
if not set(CASE_ROWS).issubset(set(profiles.simulation_row.astype(int))):
    raise ValueError("Day 9.10 profile summary is missing one or more Day 9.9 review rows.")
if max(CASE_ROWS)>=len(parameters):
    raise ValueError("A selected row is outside the Day 7 simulation table.")
PROFILE_PERIOD=profiles.set_index("simulation_row")["profile_min_period_days"].astype(float).to_dict()
if any(not np.isfinite(PROFILE_PERIOD[r]) or not 0.1 <= PROFILE_PERIOD[r] <= 13.0 for r in CASE_ROWS):
    raise ValueError("A selected Day 9.10 profile period is non-finite or outside the simulated 0.1–13 day range.")
MAP_REQUIRED={"baseline_flux","phase","depth_primary","depth_ratio","duration_fraction","ellipsoidal_amplitude","reflection_amplitude","noise_sigma"}
MAP_SEEDS={}
for r in CASE_ROWS:
    subset=map_params[map_params.simulation_row.astype(int).eq(int(r))]
    if subset.parameter.duplicated().any():
        raise ValueError(f"Duplicate MAP parameter rows for simulation row {r}.")
    seeds=subset.set_index("parameter").map_estimate.astype(float).to_dict()
    missing=MAP_REQUIRED-set(seeds)
    if missing or any(not np.isfinite(seeds[name]) for name in MAP_REQUIRED if name in seeds):
        raise ValueError(f"Invalid or missing Day 9.9 MAP seeds for row {r}: {sorted(missing)}")
    MAP_SEEDS[int(r)]=seeds
OUT=WORK/("day9-11-conditional-nuisance-nuts-"+datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
OUT.mkdir(parents=True,exist_ok=False)
CHAINS,WARMUP,SAMPLES=2,300,500
TARGET_ACCEPT,MAX_TREE_DEPTH=0.95,9
CHAIN_METHOD="parallel" if len(jax.devices())>=2 else "sequential"
print("[PASS] Day 7:",BINARY_DIR)
print("[PASS] Day 9.9:",NUTS_PATH.parent)
print("[PASS] Day 9.10 profiles:",PROFILE_PATH.parent)
print("Conditional retry rows:",CASE_ROWS)
print("Fixed periods from profile minima:",{r:PROFILE_PERIOD[r] for r in CASE_ROWS})
print(f"NUTS: {CHAINS} chains x {WARMUP} warmup + {SAMPLES} draws; diagonal mass; target_accept={TARGET_ACCEPT}; max_tree_depth={MAX_TREE_DEPTH}")
print("Output:",OUT)

In [ ]:
def smooth_clip_jax(x,sharpness=500.0):
    return (jax.nn.softplus(sharpness*x)-jax.nn.softplus(sharpness*(x-1.0)))/sharpness

def predict_jax(t,period,phase,depth,ratio,duration,ellip,reflection,baseline):
    orb=jnp.mod(t/period-phase+0.5,1.0)-0.5
    sec=jnp.mod(orb,1.0)-0.5
    half=jnp.maximum(duration/2.0,1e-6)
    ingress=jnp.maximum(half*0.18,1e-6)
    primary=smooth_clip_jax((half+ingress-jnp.abs(orb))/(2.0*ingress))
    secondary=smooth_clip_jax((half+ingress-jnp.abs(sec))/(2.0*ingress))
    return baseline-depth*primary-depth*ratio*secondary+ellip*jnp.cos(4*jnp.pi*orb)+reflection*jnp.cos(2*jnp.pi*orb)

def scalar_metrics(diag,key):
    values=[]
    for item in diag.values():
        if key in item:
            a=np.asarray(item[key])
            if a.size==1 and np.isfinite(a).all(): values.append(float(a.reshape(-1)[0]))
    return values

case_rows=[]
interval_rows=[]
plot_rows=[]
for row in CASE_ROWS:
    started=time.time()
    period_fixed=float(PROFILE_PERIOD[int(row)])
    seed_df=map_params[map_params.simulation_row.astype(int).eq(int(row))]
    seeds=MAP_SEEDS[int(row)]
    t=np.asarray(time_days,float)
    y=np.asarray(observations[int(row)],float)
    valid=np.isfinite(t)&np.isfinite(y)
    t,y=t[valid],y[valid]
    order=np.argsort(t)
    t,y=t[order],y[order]
    t=t-t.min()
    phase0=float(np.mod(seeds["phase"],1.0))
    time_j,obs_j=jnp.asarray(t),jnp.asarray(y)

    def model():
        baseline=numpyro.sample("baseline_flux",dist.Normal(1.0,0.02))
        phase_offset=numpyro.sample("phase_offset",dist.Normal(0.0,0.025))
        depth=numpyro.sample("depth_primary",dist.Uniform(0.002,0.45))
        ratio=numpyro.sample("depth_ratio",dist.Uniform(0.0,1.0))
        duration=numpyro.sample("duration_fraction",dist.Uniform(0.005,0.12))
        ellip=numpyro.sample("ellipsoidal_amplitude",dist.Uniform(0.0,0.02))
        reflection=numpyro.sample("reflection_amplitude",dist.Uniform(-0.01,0.01))
        log_noise=numpyro.sample("log_noise",dist.Uniform(np.log(1e-4),np.log(0.03)))
        phase=jnp.mod(phase0+phase_offset,1.0)
        pred=predict_jax(time_j,period_fixed,phase,depth,ratio,duration,ellip,reflection,baseline)
        numpyro.sample("flux",dist.Normal(pred,jnp.exp(log_noise)),obs=obs_j)

    init={"baseline_flux":float(np.clip(seeds["baseline_flux"],0.9501,1.0499)),
        "phase_offset":0.0,
        "depth_primary":float(np.clip(seeds["depth_primary"],0.002001,0.449999)),
        "depth_ratio":float(np.clip(seeds["depth_ratio"],1e-5,0.99999)),
        "duration_fraction":float(np.clip(seeds["duration_fraction"],0.005001,0.119999)),
        "ellipsoidal_amplitude":float(np.clip(seeds["ellipsoidal_amplitude"],1e-5,0.01999)),
        "reflection_amplitude":float(np.clip(seeds["reflection_amplitude"],-0.00999,0.00999)),
        "log_noise":float(np.log(np.clip(seeds["noise_sigma"],1.0001e-4,0.02999)))}
    kernel=NUTS(model,init_strategy=init_to_value(values=init),target_accept_prob=TARGET_ACCEPT,
                max_tree_depth=MAX_TREE_DEPTH,dense_mass=False)
    mcmc=MCMC(kernel,num_warmup=WARMUP,num_samples=SAMPLES,num_chains=CHAINS,
              chain_method=CHAIN_METHOD,progress_bar=True)
    try:
        mcmc.run(jax.random.PRNGKey(119000+int(row)),extra_fields=("diverging","accept_prob","num_steps"))
        grouped=mcmc.get_samples(group_by_chain=True)
        flat=mcmc.get_samples(group_by_chain=False)
        diag=numpyro_summary(grouped,group_by_chain=True)
        rhats,esses=scalar_metrics(diag,"r_hat"),scalar_metrics(diag,"n_eff")
        extras=mcmc.get_extra_fields(group_by_chain=True)
        divergences=int(np.asarray(extras["diverging"]).sum())
        max_rhat=max(rhats) if rhats else float("nan")
        min_ess=min(esses) if esses else float("nan")
        status="screen_pass" if np.isfinite(max_rhat) and np.isfinite(min_ess) and max_rhat<=1.05 and min_ess>=100 and divergences==0 else "needs_review"
        transformed={k:np.asarray(v) for k,v in flat.items()}
        transformed["phase"]=(phase0+transformed["phase_offset"])%1.0
        transformed["noise_sigma"]=np.exp(transformed["log_noise"])
        np.savez_compressed(OUT/f"conditional_row_{row}_posterior.npz",**transformed)
        truth=parameters.iloc[int(row)]
        truth_values={name:float(truth[name]) for name in ["phase","depth_primary","depth_ratio","duration_fraction","ellipsoidal_amplitude","reflection_amplitude","noise_sigma"]}
        truth_values["baseline_flux"]=1.0
        phase_truth=(truth_values["phase"]-phase0+0.5)%1.0-0.5
        for name in ["baseline_flux","phase_offset","depth_primary","depth_ratio","duration_fraction","ellipsoidal_amplitude","reflection_amplitude","noise_sigma"]:
            samples=transformed[name] if name!="noise_sigma" else transformed["noise_sigma"]
            q05,med,q95=np.quantile(samples,[0.05,0.5,0.95])
            target=phase_truth if name=="phase_offset" else truth_values.get(name)
            interval_rows.append({"simulation_row":int(row),"parameter":name,"period_fixed_days":period_fixed,
                "q05":float(q05),"median":float(med),"q95":float(q95),
                "known_truth_for_evaluation_only":float(target),"truth_inside_90pct_interval":bool(q05<=target<=q95),
                "posterior_scope":"conditional_on_profile_period"})
        med={k:float(np.median(v)) for k,v in transformed.items()}
        phase_fit=float((phase0+med["phase_offset"])%1.0)
        pred=np.asarray(predict_jax(time_j,period_fixed,phase_fit,med["depth_primary"],med["depth_ratio"],
            med["duration_fraction"],med["ellipsoidal_amplitude"],med["reflection_amplitude"],med["baseline_flux"]),float)
        plot_rows.append((int(row),t,y,pred,period_fixed))
        accept=np.asarray(extras["accept_prob"],float)
        steps=np.asarray(extras["num_steps"],float)
        case_rows.append({"simulation_row":int(row),"status":status,"period_fixed_profile_min_days":period_fixed,
            "max_r_hat":max_rhat,"min_effective_sample_size":min_ess,"divergences":divergences,
            "mean_accept_probability":float(np.nanmean(accept)),"median_leapfrog_steps":float(np.nanmedian(steps)),
            "seconds":round(time.time()-started,1)})
    except Exception as exc:
        case_rows.append({"simulation_row":int(row),"status":"failed","period_fixed_profile_min_days":period_fixed,
            "error":f"{type(exc).__name__}: {exc}","seconds":round(time.time()-started,1)})
    pd.DataFrame(case_rows).to_csv(OUT/"day9_11_conditional_nuisance_diagnostics.csv",index=False)
    pd.DataFrame(interval_rows).to_csv(OUT/"day9_11_conditional_nuisance_intervals.csv",index=False)
    print(f"Conditional NUTS row {row}: {case_rows[-1]}",flush=True)

In [ ]:
if plot_rows:
    ncols=2
    nrows=int(np.ceil(len(plot_rows)/ncols))
    fig,axes=plt.subplots(nrows,ncols,figsize=(14,4*nrows),squeeze=False,constrained_layout=True)
    for ax,(row,t,y,pred,period_fixed) in zip(axes.flat,plot_rows):
        ax.scatter(t,y,s=5,alpha=0.35,label="synthetic observations")
        ax.plot(t,pred,color="crimson",lw=0.9,label="conditional posterior median fit")
        ax.set_title(f"Row {row} | fixed period {period_fixed:.6f} d")
        ax.set_xlabel("Time (days)")
        ax.set_ylabel("Flux")
        ax.legend(fontsize=8)
    for ax in axes.flat[len(plot_rows):]: ax.set_visible(False)
    fig.savefig(OUT/"day9_11_conditional_nuisance_fits.png",dpi=140)
    plt.show()

case_df=pd.DataFrame(case_rows)
interval_df=pd.DataFrame(interval_rows)
coverage={name:{"n":int(len(g)),"covered":int(g.truth_inside_90pct_interval.sum()),
                "fraction":float(g.truth_inside_90pct_interval.mean())}
          for name,g in interval_df[interval_df.simulation_row.isin(case_df.loc[case_df.status.eq("screen_pass"),"simulation_row"])].groupby("parameter")} if not interval_df.empty else {}
summary={"created_utc":datetime.now(timezone.utc).isoformat(),"jax_backend":jax.default_backend(),
    "jax_devices":[str(d) for d in jax.devices()],"profile_input":str(PROFILE_PATH.parent),
    "retry_rows":CASE_ROWS,"period_conditioned_on_profile_minimum":True,
    "screen_pass_count":int(case_df.status.eq("screen_pass").sum()),
    "needs_review_or_failed_count":int(case_df.status.isin(["needs_review","failed"]).sum()),
    "conditional_nuisance_90pct_interval_coverage_evaluation_only":coverage,
    "configuration":{"chains":CHAINS,"warmup_per_chain":WARMUP,"samples_per_chain":SAMPLES,
        "target_accept_prob":TARGET_ACCEPT,"max_tree_depth":MAX_TREE_DEPTH,"dense_mass":False,"chain_method":CHAIN_METHOD},
    "limits":["Period is fixed from a data-selected profile minimum; no period posterior interval is estimated.",
        "Nuisance intervals condition on that fixed period and can understate joint uncertainty.",
        "Coverage is descriptive on four selected synthetic cases, not calibration.",
        "Known truth is evaluation-only and is not used in the fit.",
        "A screen-pass does not establish correctness for real stars."],
    "outputs":["day9_11_conditional_nuisance_diagnostics.csv","day9_11_conditional_nuisance_intervals.csv",
        "day9_11_conditional_nuisance_fits.png","conditional_row_*_posterior.npz"]}
(OUT/"day9_11_conditional_nuisance_summary.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
print(case_df.to_string(index=False))
print("Conditional nuisance coverage (evaluation-only):")
print(json.dumps(coverage,indent=2))
print(json.dumps(summary,indent=2))
print("Outputs:",OUT)